In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arjunsinghgangwar/employees-raw-data/Employees_raw_Data.csv.csv


In [2]:
#set up
Hr_data = pd.read_csv('/kaggle/input/datasets/arjunsinghgangwar/employees-raw-data/Employees_raw_Data.csv.csv')
np.random.seed(0)
Hr_data.head()

,emp_name,department,join_date,salary,age,performance_rating,city,remote_work
0,Employee_1,Engineering,2018-01-01 00:00:00,60820.29,51.0,4.0,Pune,No
1,Employee_2,NaN,2018-01-08 20:04:01,48271.19,30.0,3.0,Delhi,No
2,Employee_3,HR,2018-01-16 16:08:02,98590.99,55.0,2.0,Mumbai,No
3,Employee_4,Marketing,2018-01-24 12:12:02,71410.54,56.0,3.0,Mumbai,No
4,Employee_5,Sales,2018-02-01 08:16:03,23547.43,43.0,4.0,Chennai,yes


In [3]:
missing_values_count = Hr_data.isnull().sum()
#Hr_data.isnull().mean()*100

In [4]:
total_cells = np.prod(Hr_data.shape)
total_missing = missing_values_count.sum()
percent_missing = (total_missing / total_cells) * 100
print(percent_missing)

3.449675324675325


In [5]:
#dropping the missing values
columns_with_na_dropped = Hr_data.dropna()

In [6]:
#how much data did we lose?
print('Columns in original dataset:%d\n' % Hr_data.shape[1])
print('Columns with na dropped:%d' % columns_with_na_dropped.shape[1])

Columns in original dataset:8

Columns with na dropped:8


In [7]:
#Inconsistent data entry
import fuzzywuzzy
from fuzzywuzzy import process
import charset_normalizer
np.random.seed()
columns_with_na_dropped.head()

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


,emp_name,department,join_date,salary,age,performance_rating,city,remote_work
0,Employee_1,Engineering,2018-01-01 00:00:00,60820.29,51.0,4.0,Pune,No
2,Employee_3,HR,2018-01-16 16:08:02,98590.99,55.0,2.0,Mumbai,No
3,Employee_4,Marketing,2018-01-24 12:12:02,71410.54,56.0,3.0,Mumbai,No
4,Employee_5,Sales,2018-02-01 08:16:03,23547.43,43.0,4.0,Chennai,yes
6,Employee_7,Sales,2018-02-17 00:24:05,48087.84,38.0,5.0,Pune,Yes


In [8]:
Name = columns_with_na_dropped['department'].unique()
print(Name)

['Engineering' 'HR' 'Marketing' 'Sales' 'Finance' 'MARKETING'
 'ENGINEERING' 'SALES']


In [9]:
#Standardize department column
df = columns_with_na_dropped
df['department'] = df['department'].str.title()
df['department'] = df['department'].replace({'Hr': 'HR'})
print(df['department'].unique())

['Engineering' 'HR' 'Marketing' 'Sales' 'Finance']


/tmp/ipykernel_16/1012932586.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['department'] = df['department'].str.title()
/tmp/ipykernel_16/1012932586.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['department'] = df['department'].replace({'Hr': 'HR'})


In [10]:
#Standardized Remote_work column
df_remote = df['remote_work'].unique()
print(df_remote)

['No' 'yes' 'Yes' 'NO']


In [11]:
df['remote_work'] = df['remote_work'].str.lower()
print(df['remote_work'].unique())

['no' 'yes']


/tmp/ipykernel_16/326627733.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['remote_work'] = df['remote_work'].str.lower()


In [12]:
#Method 2 for standardizing
matches = fuzzywuzzy.process.extract('ENGINEER', 'department', limit = 10, scorer = fuzzywuzzy.fuzz.token_sort_ratio)
def replace_matches_in_column(df, department, string_to_match, min_ratio = 47):
    strings = df['department'].unique()
    matches = fuzzywuzzy.process.extract(string_to_match, strings, limit = 10, scorer = fuzzywuzzy.fuzz.token_sort_ratio)
    close_matches = [matches[0] for matches in matches if matches [1] >= min_ratio]
    rows_with_matches = df['department'].isin(close_matches)
    df.loc[row_with_matches, 'department'] = string_to_match
print(df)

         emp_name   department            join_date    salary   age  \
0      Employee_1  Engineering  2018-01-01 00:00:00  60820.29  51.0   
2      Employee_3           HR  2018-01-16 16:08:02  98590.99  55.0   
3      Employee_4    Marketing  2018-01-24 12:12:02  71410.54  56.0   
4      Employee_5        Sales  2018-02-01 08:16:03  23547.43  43.0   
6      Employee_7        Sales  2018-02-17 00:24:05  48087.84  38.0   
..            ...          ...                  ...       ...   ...   
299  Employee_300        Sales  2024-06-01 00:00:00  30092.64  40.0   
303  Employee_214        Sales  2022-07-28 02:14:51  72054.11  39.0   
304  Employee_107  Engineering  2020-04-10 15:05:25  67432.77  46.0   
306  Employee_177    Marketing  2021-10-11 03:46:21  62165.32  51.0   
307   Employee_74      Finance  2019-07-27 00:52:59  62111.89  30.0   

     performance_rating       city remote_work  
0                   4.0       Pune          no  
2                   2.0     Mumbai          no  


In [13]:
#dropping duplicates
df.drop_duplicates(inplace = True)
print(df.isnull().sum())

emp_name              0
department            0
join_date             0
salary                0
age                   0
performance_rating    0
city                  0
remote_work           0
dtype: int64


/tmp/ipykernel_16/557454460.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace = True)


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 230 entries, 0 to 299
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   emp_name            230 non-null    object 
 1   department          230 non-null    object 
 2   join_date           230 non-null    object 
 3   salary              230 non-null    float64
 4   age                 230 non-null    float64
 5   performance_rating  230 non-null    float64
 6   city                230 non-null    object 
 7   remote_work         230 non-null    object 
dtypes: float64(3), object(5)
memory usage: 16.2+ KB


In [15]:
print(df['age'].unique())

[51. 55. 56. 43. 38. 31. 53. 35. 30. 46. 25. 47. 50. 44. 27. 39. 42. 57.
 59. 58. 48. 52. 28. 23. 22. 32. 45. 26. 24. 40. 33. 37. 29. 36. 54. 34.
 41. 49.]


In [16]:
#changing age from float to integer
df['age'] = df['age'].astype(int)
print(df['age'].unique())

[51 55 56 43 38 31 53 35 30 46 25 47 50 44 27 39 42 57 59 58 48 52 28 23
 22 32 45 26 24 40 33 37 29 36 54 34 41 49]


/tmp/ipykernel_16/2815301803.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['age'] = df['age'].astype(int)


In [17]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 230 entries, 0 to 299
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   emp_name            230 non-null    object 
 1   department          230 non-null    object 
 2   join_date           230 non-null    object 
 3   salary              230 non-null    float64
 4   age                 230 non-null    int64  
 5   performance_rating  230 non-null    float64
 6   city                230 non-null    object 
 7   remote_work         230 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 16.2+ KB
None
